0. 코랩을 사용하는 경우 아래 셀을 실행하세요

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

PROJECT_PATH = '/content/drive/MyDrive/TabNet'
if not os.path.exists(PROJECT_PATH):
    os.makedirs(PROJECT_PATH)
    print(f"Folder Created: {PROJECT_PATH}")
else:
    print(f"Folder already exists: {PROJECT_PATH}")

os.makedirs(f'{PROJECT_PATH}/data', exist_ok=True)
os.makedirs(f'{PROJECT_PATH}/models', exist_ok=True)
%cd {PROJECT_PATH}

1. 주요 Requirements 설치

In [ ]:
!pip install torch pytorch-tabnet scikit-learn pandas numpy

2. TabNet 학습 코드 (데이터 전처리와 피쳐 엔지니어링 포함)

In [ ]:
import pandas as pd
import numpy as np
import torch
from pytorch_tabnet.metrics import Metric
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder
from sklearn.metrics import (
    roc_auc_score, accuracy_score, classification_report,
    precision_score, recall_score, f1_score, average_precision_score, log_loss,
    roc_curve, precision_recall_curve, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path

# ==========================================
# 1. 데이터 로더 정의, 피쳐 엔지니어링 적용 코드
# ==========================================
def apply_advanced_feature_engineering(df):
    df_new = df.copy()

    df_new['Total_Satisfaction_Score'] = (
        df_new['EnvironmentSatisfaction'] +
        df_new['JobSatisfaction'] +
        df_new['RelationshipSatisfaction'] +
        df_new['WorkLifeBalance']
    )

    df_new['Income_Per_WorkingYear'] = df_new['MonthlyIncome'] / (df_new['TotalWorkingYears'] + 1)
    df_new['Income_Per_YearAtCompany'] = df_new['MonthlyIncome'] / (df_new['YearsAtCompany'] + 1)
    df_new['Income_Per_Level'] = df_new['MonthlyIncome'] / df_new['JobLevel']
    df_new['Cost_Effectiveness'] = df_new['PercentSalaryHike'] / df_new['PerformanceRating']

    is_overtime = df_new['OverTime'].apply(lambda x: 1 if x == 'Yes' or x == 1 else 0)

    wlb_reversed = 5 - df_new['WorkLifeBalance'] #워라벨은 높을수록 좋은거였으니 번아웃 리스크 피쳐를 위해 낮은게 좋은 것으로 변환
    df_new['Burnout_Risk'] = is_overtime + wlb_reversed
    df_new['Sat_WLB_Interaction'] = df_new['Total_Satisfaction_Score'] * df_new['WorkLifeBalance']

    df_new['Promotion_Speed_Index'] = df_new['JobLevel'] / (df_new['TotalWorkingYears'] + 1)
    df_new['Stagnation_Index'] = df_new['YearsSinceLastPromotion'] / (df_new['YearsAtCompany'] + 1)
    df_new['Job_Hopping_Index'] = df_new['NumCompaniesWorked'] / (df_new['TotalWorkingYears'] + 1)

    df_new['Loyalty_Ratio'] = np.where(
        df_new['TotalWorkingYears'] > 0,
        df_new['YearsAtCompany'] / df_new['TotalWorkingYears'],
        0
    )

    return df_new

def get_hr_data_for_tabnet(filepath):
    df = pd.read_csv(filepath)
    new_df = apply_advanced_feature_engineering(df)
    df = new_df.copy()
    target_col = 'Attrition'
    X = df.drop(target_col, axis=1)
    y = LabelEncoder().fit_transform(df[target_col])

    #데이터 분할 (Train/Valid/Test = 70:15:15)
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
    X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

    # 변수 타입 분류
    cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    num_cols = X.select_dtypes(exclude=['object', 'category']).columns.tolist()

    # 수치형: 스케일링
    scaler = StandardScaler()
    X_train_num = scaler.fit_transform(X_train[num_cols])
    X_valid_num = scaler.transform(X_valid[num_cols])
    X_test_num = scaler.transform(X_test[num_cols])

    # 범주형: 오디널 인코딩
    ordinal_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_train_cat = ordinal_enc.fit_transform(X_train[cat_cols])
    X_valid_cat = ordinal_enc.transform(X_valid[cat_cols])
    X_test_cat = ordinal_enc.transform(X_test[cat_cols])

    X_train_processed = np.hstack((X_train_num, X_train_cat))
    X_valid_processed = np.hstack((X_valid_num, X_valid_cat))
    X_test_processed = np.hstack((X_test_num, X_test_cat))

    features = num_cols + cat_cols
    cat_idxs = list(range(len(num_cols), len(num_cols) + len(cat_cols)))
    cat_dims = [len(cats) for cats in ordinal_enc.categories_]

    return X_train_processed, X_valid_processed, X_test_processed, y_train, y_valid, y_test, features, cat_idxs, cat_dims

# ==========================================
# 2. 데이터 불러오기 및 커스텀 평가지표 정의
# ==========================================
filepath = '/content/drive/MyDrive/data_team7.csv'
X_train, X_valid, X_test, y_train, y_valid, y_test, features, cat_idxs, cat_dims = get_hr_data_for_tabnet(filepath)

class PRAUCMetric(Metric):
    def __init__(self):
        self._name = "pr_auc"
        self._maximize = True

    def __call__(self, y_true, y_score):
        preds_proba_1 = y_score[:, 1]
        return average_precision_score(y_true, preds_proba_1)

# ==========================================
# 3. TabNet 모델 정의 및 학습
# ==========================================
clf = TabNetClassifier(
    n_d=16, n_a=16, n_steps=2,
    gamma=1.5643739715662777,
    lambda_sparse=0.0014106348309282917,
    cat_idxs=cat_idxs,
    cat_dims=cat_dims,
    cat_emb_dim=2,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=0.049689515723358134),
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    scheduler_params={"step_size":10, "gamma":0.9},
    mask_type='sparsemax',
    device_name='auto'
)
print("TabNet 학습 시작")
clf.fit(
    X_train=X_train, y_train=y_train,
    eval_set=[(X_train, y_train), (X_valid, y_valid)],
    eval_name=['train', 'valid'],
    eval_metric=[PRAUCMetric],
    max_epochs=100,
    patience=100,
    batch_size=256,
    virtual_batch_size=128,
    num_workers=0,
    drop_last=False
)

# ==========================================
# 4. 모델 평가 및 피처 중요도 확인
# ==========================================
preds = clf.predict(X_test)
preds_proba = clf.predict_proba(X_test)[:, 1]

precision = precision_score(y_test, preds)
recall = recall_score(y_test, preds)
f1 = f1_score(y_test, preds)
roc_auc = roc_auc_score(y_test, preds_proba)
pr_auc = average_precision_score(y_test, preds_proba)
logloss = log_loss(y_test, preds_proba)

print("\n[최종 모델 성능 평가 지표 (6 Metrics)]")
print("-" * 40)
print(f"1. Precision: {precision:.4f} | 2. Recall: {recall:.4f} | 3. F1-Score: {f1:.4f}")
print(f"4. ROC-AUC  : {roc_auc:.4f} | 5. PR-AUC: {pr_auc:.4f} | 6. Log Loss: {logloss:.4f}")
print("-" * 40)

3. 학습된 모델의 성능을 시각화한 그래프 4가지 pdf로 저장

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_curve, average_precision_score, confusion_matrix
import pandas as pd
import numpy as np

sns.set_theme(style="whitegrid")
plt.rcParams['axes.unicode_minus'] = False

def save_final_4_plots(y_test, preds, preds_proba, history_train, history_valid, best_iteration, feature_names, importances):
    # ==========================================
    # 1. Learning Curve
    # ==========================================
    fig1 = plt.figure(figsize=(10, 6))

    epochs = range(1, len(history_train) + 1)
    plt.plot(epochs, history_train, label='Train PR-AUC', color='#4C72B0', lw=2)
    plt.plot(epochs, history_valid, label='Validation PR-AUC', color='#DD8452', lw=2)

    plt.axvline(x=best_iteration, color='red', linestyle='--', lw=2, label=f'Best Model (Epoch {best_iteration})')
    plt.text(best_iteration + 1, max(history_valid) * 0.95, 'Optimal Point', color='red', fontweight='bold')

    plt.title("Learning Curve", fontsize=16, fontweight='bold')
    plt.xlabel("Epochs", fontsize=13)
    plt.ylabel("PR-AUC Score", fontsize=13)
    plt.legend(loc='lower right')
    plt.tight_layout()

    fig1.savefig('Learning_Curve_BestModel.pdf', bbox_inches='tight')
    plt.show()
    plt.close(fig1)

    # ==========================================
    # 2. PR-AUC Curve
    # ==========================================
    fig2 = plt.figure(figsize=(8, 6))

    pr_auc_score = average_precision_score(y_test, preds_proba)
    precisions, recalls, _ = precision_recall_curve(y_test, preds_proba)

    plt.plot(recalls, precisions, color='purple', lw=3, label=f'Model PR-AUC = {pr_auc_score:.4f}')

    plt.title("PR Curve", fontsize=16, fontweight='bold')
    plt.xlabel("Recall", fontsize=13)
    plt.ylabel("Precision", fontsize=13)
    plt.legend(loc='upper right')
    plt.tight_layout()

    fig2.savefig('PR_AUC_Curve.pdf', bbox_inches='tight')
    plt.show()
    plt.close(fig2)

    # ==========================================
    # 3. Confusion Matrix
    # ==========================================
    fig3 = plt.figure(figsize=(7, 6))

    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', annot_kws={"size": 16, "weight": "bold"})

    plt.title("Confusion Matrix", fontsize=16, fontweight='bold')
    plt.xlabel("Predicted Label", fontsize=13)
    plt.ylabel("True Label", fontsize=13)
    plt.xticks(ticks=[0.5, 1.5], labels=['Stay', 'Attrition'], fontsize=11)
    plt.yticks(ticks=[0.5, 1.5], labels=['Stay', 'Attrition'], fontsize=11, rotation=0)
    plt.tight_layout()

    fig3.savefig('Confusion_Matrix.pdf', bbox_inches='tight')
    plt.show()
    plt.close(fig3)

    # ==========================================
    # 4. Top 20 Feature Importance
    # ==========================================
    fig4 = plt.figure(figsize=(10, 8))

    importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
    importance_df = importance_df.sort_values(by='Importance', ascending=False).head(20)

    sns.barplot(x='Importance', y='Feature', data=importance_df, palette='viridis')

    plt.title("Top 20 Feature Importances", fontsize=16, fontweight='bold')
    plt.xlabel("Importance Score", fontsize=13)
    plt.ylabel("Features", fontsize=13)
    plt.tight_layout()

    fig4.savefig('Feature_Importance_Top20.pdf', bbox_inches='tight')
    plt.show()
    plt.close(fig4)

save_final_4_plots(
    y_test = y_test,
    preds = clf.predict(X_test),
    preds_proba = clf.predict_proba(X_test)[:, 1],
    history_train = clf.history['train_pr_auc'],
    history_valid = clf.history['valid_pr_auc'],
    best_iteration = clf.best_epoch,
    feature_names = features,
    importances = clf.feature_importances_
)

4. 하이퍼 파라미터 튜닝을 위한 optuna 라이브러리 설치

In [ ]:
!pip install optuna

5. 선정한 평가지표(PR-AUC)를 기준으로 최적의 하이퍼 파라미터를 알려주는 코드

In [ ]:
import optuna
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, recall_score
import torch
from pytorch_tabnet.tab_model import TabNetClassifier
from pytorch_tabnet.metrics import Metric

# ==========================================
# 1.평가지표 정의
# ==========================================
class PRAUCMetric(Metric):
    def __init__(self):
        self._name = "pr_auc"
        self._maximize = True

    def __call__(self, y_true, y_score):
        preds_proba_1 = y_score[:, 1]
        return average_precision_score(y_true, preds_proba_1)

TARGET_METRIC = 'pr_auc'

# ==========================================
# 2. Objective Function 정의
# ==========================================
def objective(trial):
    n_steps = trial.suggest_int('n_steps', 2, 5)
    n_d = trial.suggest_int('n_d', 8, 24, step=8) # 8, 16, 24, 32
    n_a = n_d
    gamma = trial.suggest_float('gamma', 1.0, 2.0)
    lambda_sparse = trial.suggest_float('lambda_sparse', 1e-5, 1e-2, log=True)
    lr = trial.suggest_float('lr', 0.005, 0.05, log=True)

    clf = TabNetClassifier(
        n_d=n_d, n_a=n_a, n_steps=n_steps,
        gamma=gamma,
        lambda_sparse=lambda_sparse,
        cat_idxs=cat_idxs, cat_dims=cat_dims, cat_emb_dim=2,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=lr),
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        scheduler_params={"step_size":10, "gamma":0.9},
        mask_type='sparsemax',
        device_name='auto',
        verbose=0
    )

    clf.fit(
        X_train=X_train, y_train=y_train,
        eval_set=[(X_valid, y_valid)],
        eval_name=['valid'],
        eval_metric=[PRAUCMetric] if TARGET_METRIC == 'pr_auc' else ['auc'],
        max_epochs=100,
        patience=30,
        batch_size=256,
        virtual_batch_size=128,
        num_workers=0,
        drop_last=False
    )

    if TARGET_METRIC in ['roc_auc', 'pr_auc']:
        preds_proba = clf.predict_proba(X_valid)[:, 1]
        if TARGET_METRIC == 'roc_auc':
            score = roc_auc_score(y_valid, preds_proba)
        elif TARGET_METRIC == 'pr_auc':
            score = average_precision_score(y_valid, preds_proba)

    elif TARGET_METRIC in ['f1', 'recall']:
        preds = clf.predict(X_valid)
        if TARGET_METRIC == 'f1':
            score = f1_score(y_valid, preds)
        elif TARGET_METRIC == 'recall':
            score = recall_score(y_valid, preds)

    return score

# ==========================================
# 3. Optuna 실행
# ==========================================
print(f"Optuna 하이퍼파라미터 튜닝을 시작 (목표: {TARGET_METRIC.upper()})")

study = optuna.create_study(direction='maximize', study_name=f"TabNet_{TARGET_METRIC}_Tuning")

study.optimize(objective, n_trials=150)

print(f"[튜닝 완료] {TARGET_METRIC} 기준 최고 점수: {study.best_value:.4f}")
print("베스트 파라미터:", study.best_params)